# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The FAIR^2 dataset is represented in Croissant schema and is available at the following URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show basic metadata
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their reference `@id`s from the dataset.

In [ ]:
# List all record sets available in the Croissant schema
print("Available record sets (by '@id'):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '(no name)')}")

# Choose a record set to inspect its fields and example records
# We expect a record set with proteins and their features; let's inspect all
for rs in record_sets:
    print(f"\nRecord set '@id': {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (by '@id'):")
    for f in fields:
        print(f"   - {f['@id']}: {f.get('name', f['@id'])}")
    print(f"Example records from {rs['@id']}: ")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs['@id'])):
            print(rec)
            if i >= 2:
                break
    except Exception as e:
        print(f"  (Could not retrieve records: {e})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
All entities (record set, field, column) are referenced by their `@id`.

In [ ]:
# List of record set @id's (update this list after inspecting section 2 output)
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nRecord set '{record_set_id}': loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print(f"\nRecord set '{record_set_id}': No records returned.")
    except Exception as e:
        print(f"\nError loading records for '{record_set_id}': {e}")

# For downstream analysis, pick the main record set (update if necessary)
main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id:
    display_cols = dataframes[main_record_set_id].columns.tolist()
    print(f"\nMain record set columns ({main_record_set_id}):\n{display_cols}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's apply common EDA steps. 
- Filter records by a chosen numeric field (`@id`).
- Normalize that field.
- Group by a categorical field, if present.

All fields should be referenced by `@id` as established above.

In [ ]:
# Set the main record set and candidate fields by `@id`
record_set_id = main_record_set_id
df = dataframes.get(record_set_id)
if df is not None and not df.empty:
    # Inspect columns and select a numeric field and a group by field
    print(f"Fields (@id's): {df.columns.tolist()}")
    # Guess a numeric field (replace with real @id as needed)
    # Examples (you may need to adjust): 'cr:field:abundance', 'cr:field:mw', etc.
    numeric_fields = [col for col in df.columns if df[col].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields detected: {numeric_fields}")
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered {filtered_df.shape[0]} records with {numeric_field_id} > {threshold:.2f}")
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try grouping by a likely categorical field
        group_by_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
        if group_by_candidates:
            group_by_field = group_by_candidates[0]
            grouped = filtered_df.groupby(group_by_field)[numeric_field_id].mean().sort_values(ascending=False)
            print(f"\nGrouped mean {numeric_field_id} by {group_by_field} (@id):")
            print(grouped.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields available for analysis.")
else:
    print("DataFrame not available or empty for analysis.")

## 5. Visualization
Visualize data distributions and feature relationships.
All references to fields use their `@id` as in earlier sections.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # If grouped means prepared in EDA
    if 'grouped' in locals() and not grouped.empty:
        grouped.head(10).plot(kind='bar', figsize=(10,4))
        plt.title(f'Mean {numeric_field_id} by {group_by_field}')
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. We referenced all Croissant entities via their `@id`, ensuring reproducibility and schema adherence. Further analysis can be performed by referencing relevant record sets or fields by their `@id` as shown.